In [ ]:
from torch.nn import *
from functools import partial
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def center(a):
    return a.mul(2.0).add_(-1.0)


def dat(k, c, s=1, n=1, l=1, rx=1.0, rk=1.0, rb=1.0):
    l = 2 ** int(np.log2(c * s * (k + 1) * l))
    y = torch.randint(c * s, (n, l))
    return (
        torch.randn(n, c * s, k)
        .mul_(rx * rb * l ** (1 / k))[torch.arange(n)[:, None], y]
        .view(n * l, 1, k)
        .baddbmm(
            torch.randn(n * l, 1, k).mul_(rx),
            torch.randn(n, c * s, k, k)[torch.arange(n)[:, None], y].view(n * l, k, k),
        )
        .view(n, l, k),
        unpack(y // s if s - 1 & s else y >> int(np.log2(s)), np.log2(c).__ceil__()),
    )


class F(Module):
    def __init__(s, f, *a):
        super().__init__()
        s.f = f
        s.a = a

    def forward(s, *x):
        return s.f(*x, *s.a)

    def extra_repr(s):
        return s.f.__name__ + str(s.a)


class Linears(Module):
    def __init__(s, i, o, n=1, b=None):
        super().__init__()
        s.w = Parameter(init.kaiming_uniform_(torch.empty(n, o, i)).transpose(1, 2))
        s.b = b and Parameter(init.uniform_(torch.empty(n, 1, o), -(i ** 0.5), i ** 0.5))

    def forward(s, x):
        x = x @ s.w
        return x if s.b is None else x + s.b

    def extra_repr(s):
        return f"{s.w.size(1)}, {s.w.size(2)}, {s.w.size(0)}, {s.w is not None}"


def Nn(i, h, n=1, L=Linears, R=Linears):
    return Sequential(L(i, h, n), Hardswish(1), R(h, h, n))


class Relate(Module):
    def __init__(s, l, r, h, n=1, L=Linears, R=partial(Linears, b=0), M=lambda *_: F(torch.add), O=lambda *_: Hardswish(1)):
        super().__init__()
        s.L = L(l, h, n)
        s.R = R(r, h, n)
        s.M = M(l, r, h, n)
        s.O = O(h, h, n)

    def forward(s, l, r):
        return s.O(s.M(s.L(l), s.R(r)))


class Attention(Module):
    def __init__(s, q, k, v, h, n=1, Q=Nn, K=partial(Relate, O=partial(Nn, L=Identity)), V=Relate, M=lambda *_: F(lambda l, r: l @ r.transpose(-2, -1))):
        super().__init__()
        s.Q = Q(q, h, n)
        s.K = K(k, v, h, n)
        s.V = V(k, v, h, n)
        s.M = M(h, h, h, n)

    def forward(s, q, k, v):
        return s.M(s.Q(q), s.K(k, v)).softmax(-1) @ s.V(k, v)


class TrainEval(Module):
    def __init__(s, q, k, v, o, l=1, h=1, n=1, A=Attention, B=Attention, C=Relate):
        super().__init__()
        l = int(l * 2 ** o)
        h = int(h * o)
        L = np.log2(l).__ceil__()
        s.i = center(unpack(torch.arange(l), L))
        s.A = A(L, k, v, h, n)
        s.B = B(q, L, h, h, n)
        s.C = C(q, h, h, n)
        s.D = Linear(h, o)

    def forward(s, q, k, v):
        q = q[..., None, :, :]
        k = k[..., None, :, :]
        v = v[..., None, :, :]
        return s.D(s.C(q, s.B(q, s.i, s.A(s.i, k, v))).sum(-3))

In [ ]:
n = 4
m = TrainEval(n, n, n, n)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.01, 10000, pct_start=0.01, div_factor=9, final_div_factor=99)
for i in range(O.total_steps // 500):
    l = []
    a = []
    for i in range(5):
        for i in range(100):
            x, y = dat(n, 1 << n, n=64)
            x = m(x, x, y)
            i = functional.binary_cross_entropy_with_logits(x, y)
            i.backward()
            o.step()
            O.step()
            o.zero_grad()
        x = (x.signbit() != y).all(-1)
        l.append(i)
        a.append(x.sum() / x.numel())
    print(sum(l) / len(l), sum(a) / len(a))

In [ ]:
n = 4
m = TrainEval(n, n, n, n)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.01, 10000, pct_start=0.01, div_factor=9, final_div_factor=99)
for i in range(O.total_steps // 500):
    l = []
    a = []
    for i in range(5):
        for i in range(100):
            x, y = dat(n, 1 << n, n=64)
            x = m(x, x, y)
            i = functional.binary_cross_entropy_with_logits(x, y)
            i.backward()
            o.step()
            O.step()
            o.zero_grad()
        x = (x.signbit() != y).all(-1)
        l.append(i)
        a.append(x.sum() / x.numel())
    print(sum(l) / len(l), sum(a) / len(a))